# Phase 2: Split strategy

GitHub issue: #3. Depends on #1 (target) and #2 (dataset, and its correction - the split rule this
notebook locks was fixed there after a leakage bug).

In [1]:
import json
from pathlib import Path

import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FROZEN_DIR = ROOT / "data" / "frozen"

manifest = json.loads((FROZEN_DIR / "manifest.json").read_text())
print(json.dumps(manifest["split_cutoffs_ms"], indent=2))
print(json.dumps(manifest["example_counts"], indent=2))

{
  "ts_min": 1659304800025,
  "train_end": 1660514400025,
  "val_end": 1661119200025
}
{
  "test": 639037,
  "train": 512161,
  "val": 396786
}


## Step 1: Split type and justification

Time-based, keyed off each example's own `label_ts` (the held-out event's timestamp) - never session
identity or a session's first event. Two reasons: AGENTS.md's own rule ("split by time only"), and the
standard reason a model that scores future behavior must never see anything timestamped after what
val/test will be scored on.

Session identity was tried first and rejected: OTTO sessions can span the entire 4-week window (Phase 1
notebook, Step 3), so grouping by session start mixed weeks together.

## Step 2: No entity leakage across splits

Each session contributes exactly one row to `ranking_examples.parquet` (its own leave-last-out
example), so no session's *label* appears in more than one split by construction. A session's prefix
can span a long real history before its own label_ts - that's the user's own legitimate past, not a
cross-split leak, since nothing after `label_ts` is ever used for that row.

In [2]:
examples = pl.read_parquet(FROZEN_DIR / "ranking_examples.parquet")

from datetime import datetime, timezone

def to_date(ms):
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).date()

cutoffs = manifest["split_cutoffs_ms"]
print("ts_min:", to_date(cutoffs["ts_min"]))
print("train_end:", to_date(cutoffs["train_end"]))
print("val_end:", to_date(cutoffs["val_end"]))

ts_min: 2022-07-31
train_end: 2022-08-14
val_end: 2022-08-21


## Step 3: date range, row count, label rate per split

In [3]:
summary = (
    examples.group_by("split")
    .agg(
        pl.len().alias("n"),
        (pl.col("label_grade") == 1).mean().alias("click_share"),
        (pl.col("label_grade") == 2).mean().alias("cart_share"),
        (pl.col("label_grade") == 3).mean().alias("order_share"),
    )
    .sort("split")
)
summary

split,n,click_share,cart_share,order_share
str,u32,f64,f64,f64
"""test""",639037,0.920418,0.037744,0.041838
"""train""",512161,0.914988,0.038,0.047013
"""val""",396786,0.921454,0.036415,0.042131


**Finding**: test is the largest split by row count (639,037) despite covering the same 1-week span
as val (396,786). This is expected under right-censoring, not an anomaly: many long-running sessions
are still active when the 4-week collection window ends, so their last-*observed* event clusters near
the end of the window rather than reflecting where the session actually ended. Label rates (click/cart/
order mix) are stable across all three splits (see the table above), so this isn't a hidden distribution
shift - just a volume effect from how "last observed event" behaves near a truncated window.

## Step 4: Lock the test set

`split == "test"` (639,037 examples, from `val_end` onward) is locked as of this notebook. It is not
used for EDA (Phase 3), feature selection (Phase 4), baseline tuning (Phase 5), or model tuning
(Phase 6) - only for the single Phase 7 evaluation. Every notebook from here on that touches `test`
says so explicitly and does not report or act on any test-derived number before Phase 7.

## Options considered

- Splitting by session's first event (the original Phase 1 attempt): rejected - leaks future weeks
  into "training" data, see Phase 1 notebook's correction.
- A single train/test split with cross-validation folds inside train, instead of a fixed val slice:
  rejected - a fixed time-ordered val slice matches "default to time based for any model that will
  score future data" more directly than CV folds, which would mix time periods within a fold.
- Equal-width week boundaries other than 2/1/1: not explored beyond the one used - no strong reason to
  prefer another split of the 4 weeks, and this matches what the archived plan already used
  successfully.

## Exit criteria

Split definitions saved against the snapshot version (`data/frozen/manifest.json`'s
`split_cutoffs_ms`), test set locked (Step 4 above). Met.